In [0]:
%run "../01-setup/1.configure_access_to_cloud_storage"

In [0]:
%run "../01-setup/2.common_functions"

In [0]:
dbutils.widgets.text("p_data_source", "")
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
dbutils.widgets.text("p_file_date", "2025-01")
v_file_date = dbutils.widgets.get("p_file_date")
raw_race_path = f"{raw_folder_path}/{v_file_date}"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
from pyspark.sql.functions import current_timestamp

In [0]:
pitstops_schema = StructType([
  StructField("raceId", IntegerType(), False),
  StructField("driverId", IntegerType(), False),
  StructField("stop", IntegerType(), False),
  StructField("lap", IntegerType(), False),
  StructField("time", StringType(), False),
  StructField("duration", StringType(), True),
  StructField("milliseconds", IntegerType(), True)
])
pitstops_df = spark.read \
  .schema(pitstops_schema) \
  .option("multiline", True) \
  .json(f"{raw_race_path}/pit_stops.json")
display(pitstops_df)


In [0]:
final_pitstops_df = pitstops_df \
  .withColumnRenamed("raceID", "race_id") \
  .withColumnRenamed("driverId", "driver_id") \
  .withColumn("ingestion_date", current_timestamp())
display(final_pitstops_df)

In [0]:
final_pitstops_df.write.mode("overwrite").parquet(f"{processed_folder_path}/pit_stops")

In [0]:
df = spark.read.parquet(f"{processed_folder_path}/pit_stops")
display(df)

In [0]:
dbutils.notebook.exit("Success")